# Quadruped Locomotion (Go2): 보상 단계별 이어서 학습

전체 학습(2,000,000 step)은 한 번에 다 돌리려면 오래 걸립니다. 그래서 각 보상 단계로
**거의 끝까지(~2,000,000 step) 미리 학습해 둔 체크포인트**를 불러와, 여기서는
**4,800 step 만 이어서** 학습하며 결과를 확인합니다.

이 노트북은 보상 설정이 단계적으로 어떻게 바뀌는지 설명하고, 각 단계 모델을 이어서 학습합니다.

- **라운드 1** : `env1` (기본 task-only 보상) → `pretrained_env1` 를 `envs1.yaml` 로 4,800 step 이어서 학습
- **라운드 2** : `env1 → env2` 보상 변화 설명 → `pretrained_env2` 를 `envs2.yaml` 로 4,800 step 이어서 학습
- **라운드 3** : `env2 → env3` 보상 변화 설명 → `pretrained_env3` 를 `envs3.yaml` 로 4,800 step 이어서 학습


## 실습 2 — Ablation (보상 단계별 비교)

각 실험은 아래 4단계로 진행합니다.

| 단계 | 함수 | 내용 |
|------|------|------|
| **①** | `reward_diff()` | 이전 ↔ 이번 yaml 의 바뀐 항목 출력 |
| **②** | `finetune()` | pretrained 모델에서 N step 추가 학습 |
| **③** | `rollout_and_video()` | command [0.7, 0, 0] 보행 영상 생성 |
| **④** | 비교 | 무엇이 달라졌는가 |


| 실험 | checkpoint | 보상 설정 |
|------|----------------|-----------|
| **Exp 1** | `pretrained_env1` | `envs1.yaml`: 속도만 추종 |
| **Exp 2** | `pretrained_env2` | `envs2.yaml`: 정규화·gait 페널티 |
| **Exp 3** | `pretrained_env3` | `envs3.yaml`: positive shaping |

---

## 0. 환경 설정 (오프라인)


- `mujoco`, `stable-baselines3`, `gymnasium`, `torch`, `imageio[ffmpeg]`, `pygments`, `tensorboard`
- 검증 환경 예: numpy 1.26.4 / gymnasium 0.29.1 / stable-baselines3 2.3.0 / mujoco 3.8.0


## 0. 의존성 설치

레포의 `requirements.txt` 를 현재 커널 환경에 설치합니다. 이미 설치돼 있으면 건너뛰어도 됩니다.

In [26]:
# 의존성 설치 (현재 커널의 파이썬에 requirements.txt 설치)
import os, sys

# requirements.txt 가 있는 레포 루트를 위로 탐색
_root = os.getcwd()
for _ in range(6):
    if os.path.isfile(os.path.join(_root, "requirements.txt")):
        break
    _parent = os.path.dirname(_root)
    if _parent == _root:
        break
    _root = _parent
req = os.path.join(_root, "requirements.txt")
assert os.path.isfile(req), f"requirements.txt 를 못 찾음 (cwd={os.getcwd()})"
print("installing:", req)

# GPU(CUDA)면 torch CUDA 빌드를 먼저 설치 (requirements 의 torch 는 기본/CPU 빌드)
import subprocess
try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
    print("GPU 감지됨 → torch CUDA(cu117) 빌드 설치")
    get_ipython().system(f'{sys.executable} -m pip install -q torch==2.0.1 --index-url https://download.pytorch.org/whl/cu117')
except Exception:
    print("GPU 미감지 → 기본(CPU) torch 사용")

get_ipython().system(f'{sys.executable} -m pip install -q -r {req}')
print("설치 완료")

installing: c:\Users\enban\OneDrive\Desktop\RL_tutorial-main\requirements.txt
GPU 미감지 → 기본(CPU) torch 사용
설치 완료


---

In [27]:
# 오프라인(로컬 Jupyter) 환경 설정
# 사전 준비: mujoco, stable-baselines3, gymnasium, torch, imageio[ffmpeg],
#            pygments, tensorboard 가 설치된 로컬 환경에서 레포 안에서 실행하세요. 
import os, sys
import yaml
import torch  

in_colab = False
repo_dir = os.getcwd()
for _ in range(6):
    if os.path.isdir(os.path.join(repo_dir, "src")) and os.path.isdir(os.path.join(repo_dir, "unitree_go2")):
        break
    parent = os.path.dirname(repo_dir)
    if parent == repo_dir:
        break
    repo_dir = parent
assert os.path.isdir(os.path.join(repo_dir, "src")), (
    f"레포 루트를 못 찾음 (cwd={os.getcwd()}). 레포 폴더 안에서 노트북을 실행하세요.")
os.chdir(repo_dir)
sys.path.insert(0, repo_dir)
sys.path.insert(0, os.path.join(repo_dir, "src"))

import platform
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
_SYS = platform.system()  # 'Linux' | 'Windows' | 'Darwin'


if _SYS == "Windows":
    RENDER_GL_ORDER = ["wgl", "glfw"]
elif _SYS == "Darwin":
    RENDER_GL_ORDER = ["cgl", "glfw"]
else:  # Linux
    RENDER_GL_ORDER = ["egl", "osmesa"] if HAS_GPU else ["osmesa", "egl"]

os.environ["MUJOCO_GL"] = "disable"
print("repo_dir:", repo_dir, "| OS:", _SYS, "| device:", DEVICE,
      ("(GPU)" if HAS_GPU else "(CPU)"), "| render GL:", RENDER_GL_ORDER)

repo_dir: c:\Users\enban\OneDrive\Desktop\RL_tutorial-main | OS: Windows | device: cpu (CPU) | render GL: ['wgl', 'glfw']


In [28]:
from pathlib import Path
from IPython.display import HTML, display
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import HtmlFormatter
import inspect

def _render_code(code, title="code", max_height=400, bg="transparent", indent=16):
    style_name = "native" if in_colab else "friendly"
    formatter = HtmlFormatter(style=style_name, noclasses=True, linenos="inline")
    html = highlight(code, PythonLexer(), formatter)
    css = """
    <style>
    .highlight pre { margin: 0; text-align: left; }
    </style>
    """
    return HTML(f"""
    {css}
    <details>
      <summary>{title}</summary>
      <div style="margin-top:8px; margin-left:{indent}px; max-height:{max_height}px; overflow:auto; border:1px solid #ddd; padding:10px; background:{bg};">
        {html}
      </div>
    </details>
    """)


def show_code(path, max_height=400, bg="transparent"):
    code = Path(path).read_text()
    return _render_code(code, title=str(path), max_height=max_height, bg=bg)

def show_func(obj, max_height=400, bg="transparent"):
    code = inspect.getsource(obj)
    return _render_code(code, max_height=max_height, bg=bg)

---

## 1. 설정 파일 살펴보기

- **`src/params.yaml`** — PPO 하이퍼파라미터/학습 설정
- **`src/mdp/reward.py`** — 보상 함수 구현

보상 가중치(`envsN.yaml`)는 각 라운드 상단에서 해당 단계의 것만 펼쳐 봅니다.


In [29]:
display(show_code(f"{repo_dir}/src/params.yaml"))
display(show_code(f"{repo_dir}/src/mdp/reward.py", max_height=600))

---

## 2. 공통 헬퍼 정의

세 라운드에서 반복 사용할 함수와 설정을 정의합니다.

- 상단 설정: `ROUND1_MODEL`/`ROUND1_CFG` ~ `ROUND3_MODEL`/`ROUND3_CFG`, `ADDITIONAL_TIMESTEPS`
- `reward_diff(a, b)` — 두 envs.yaml 사이에 바뀐 보상/설정 항목을 출력
- `rollout_and_video(model, cfg, tag)` — 모델을 롤아웃해 mp4 저장 (test 로직 이식)
- `show_video(path)` — 노트북에서 영상 재생
- `finetune(model_in, cfg, tag)` — 해당 보상 설정으로 4,800 step 이어서 학습


In [ ]:
import time, gc, shutil
import numpy as np
import imageio
from tqdm.auto import tqdm

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import (
    EvalCallback, CheckpointCallback, CallbackList,
)
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv

import src.go2_mujoco_env as go2_env
from src.utils.reward_logging_callback import RewardLoggingCallback
from IPython.display import Video

with open(f"{repo_dir}/src/params.yaml", "r", encoding="utf-8") as f:
    policy_cfg = yaml.safe_load(f)


ROUND1_MODEL = f"{repo_dir}/models/pretrained_env1/best_model.zip"
ROUND1_CFG   = f"{repo_dir}/src/envs1.yaml"
ROUND2_MODEL = f"{repo_dir}/models/pretrained_env2/best_model.zip"
ROUND2_CFG   = f"{repo_dir}/src/envs2.yaml"
ROUND3_MODEL = f"{repo_dir}/models/pretrained_env3/best_model.zip"
ROUND3_CFG   = f"{repo_dir}/src/envs3.yaml"
ADDITIONAL_TIMESTEPS = 4800  
N_ENVS = 12 
SEED = policy_cfg["seed"]

for _m in (ROUND1_MODEL, ROUND2_MODEL, ROUND3_MODEL):
    assert os.path.exists(_m), f"모델을 찾을 수 없습니다: {_m}"

VIDEO_DIR = f"{repo_dir}/models/_reward_tuning_videos"
os.makedirs(VIDEO_DIR, exist_ok=True)


def rollout_and_video(model_path, env_cfg_path, tag):
    import subprocess, sys
    video_path = f"{VIDEO_DIR}/rollout_{tag}.mp4"
    base = [
        sys.executable, "-u", f"{repo_dir}/src/render_rollout.py",
        "--prj", repo_dir, "--model", model_path, "--cfg", env_cfg_path,
        "--out", video_path, "--command", "0.7", "0.0", "0.0",
        "--max_time_s", str(policy_cfg["test"]["max_time_step_s"]),
        "--width", "320", "--height", "240", "--camera", "tracking",
    ]
    for gl in RENDER_GL_ORDER:
        res = subprocess.run(base + ["--gl", gl], capture_output=True, text=True)
        if res.returncode == 0 and os.path.exists(video_path):
            print(f"[{tag}] 라이브 렌더 성공 ({gl})")
            return video_path
    fb = f"{repo_dir}/models/pretrained_{tag.split('_')[0]}/rollout_pretrained_{tag.split('_')[0]}.mp4"
    if os.path.exists(fb):
        print(f"ℹ️ [{tag}] 라이브 렌더 미지원 환경 → 사전 렌더링 영상 표시: {os.path.basename(fb)}")
        return fb
    print(f"⚠️ [{tag}] 렌더 실패 + 대체 영상 없음")
    return video_path


def show_video(video_path):
    display(Video(video_path, embed=True, html_attributes="controls autoplay loop"))



def finetune(model_in_path, env_cfg_path, run_tag):
    log_dir = f"{repo_dir}/logs"
    os.makedirs(log_dir, exist_ok=True)
    env_kwargs = {"prj_path": repo_dir, "cfg_path": env_cfg_path}
    vec_env = make_vec_env(go2_env.Go2MujocoEnv, env_kwargs=env_kwargs,
                           n_envs=N_ENVS, seed=SEED, vec_env_cls=SubprocVecEnv)
    eval_env = make_vec_env(go2_env.Go2MujocoEnv, env_kwargs=env_kwargs,
                            n_envs=1, seed=SEED + 10_000, vec_env_cls=DummyVecEnv)

    run_name = time.strftime("%Y-%m-%d_%H-%M-%S") + f"-{run_tag}"
    model_path = f"{repo_dir}/models/{run_name}"
    os.makedirs(model_path, exist_ok=True)
    shutil.copy2(env_cfg_path, f"{model_path}/envs.yaml")   
    print("저장 위치:", model_path)

    _dummy = go2_env.Go2MujocoEnv(prj_path=repo_dir, cfg_path=env_cfg_path, render_mode=None)
    custom_objects = {
        "observation_space": _dummy.observation_space,
        "action_space": _dummy.action_space,
        "lr_schedule": lambda _: policy_cfg["policy"]["learning_rate"],
        "clip_range": lambda _: policy_cfg["policy"]["clip_range"],
    }
    _dummy.close()

    callbacks = CallbackList([
        EvalCallback(eval_env, best_model_save_path=model_path, log_path=log_dir,
                     eval_freq=max(policy_cfg["eval_freq"] // N_ENVS, 1),
                     n_eval_episodes=5, deterministic=True, render=False),
        CheckpointCallback(
            save_freq=max(policy_cfg["policy"]["n_steps"] * policy_cfg["log"]["interval"] // N_ENVS, 1),
            save_path=model_path, name_prefix="model",
            save_replay_buffer=False, save_vecnormalize=False),
        RewardLoggingCallback(),
    ])

    print(f"[{run_tag}] load pretrained: {model_in_path}")
    model = PPO.load(model_in_path, env=vec_env, custom_objects=custom_objects,
                     verbose=1, tensorboard_log=log_dir, device=DEVICE)
    model.learning_rate = policy_cfg["policy"]["learning_rate"]
    model._setup_lr_schedule()
    model.learn(total_timesteps=ADDITIONAL_TIMESTEPS, reset_num_timesteps=False,
                progress_bar=False, tb_log_name=run_name, callback=callbacks)
    model.save(f"{model_path}/final_model")

    vec_env.close(); eval_env.close()
    del model; gc.collect()

    out = f"{model_path}/best_model.zip"
    if not os.path.exists(out):
        out = f"{model_path}/final_model.zip"
    print(f"[{run_tag}] saved ->", out)
    return out


def reward_diff(cfg_a_path, cfg_b_path):
    a = yaml.safe_load(open(cfg_a_path, encoding="utf-8"))
    b = yaml.safe_load(open(cfg_b_path, encoding="utf-8"))
    rows = []
    def walk(da, db, prefix=""):
        for k in dict.fromkeys(list(da) + list(db)):
            va, vb = da.get(k), db.get(k)
            if isinstance(va, dict) or isinstance(vb, dict):
                walk(va or {}, vb or {}, prefix + k + ".")
            elif va != vb:
                rows.append((prefix + k, va, vb))
    walk(a, b)
    print(f"보상/설정 변화: {os.path.basename(cfg_a_path)} -> {os.path.basename(cfg_b_path)}")
    for key, va, vb in rows:
        print(f"  {key:26s}: {va}  ->  {vb}")
    if not rows:
        print("  (변경 없음)")
    return rows


# 비교용 영상 목록
round_videos = []

---

## 3. 라운드 1 — env1 (기본 task-only 보상)

`env1` 은 추가 보상 없이 **속도 추종(`linear/angular_vel_tracking`)만** 보는 가장 기본적인
*task-only* 설정입니다. `env1` 보상으로 ~2,000,000 step 미리 학습해 둔 체크포인트
(`pretrained_env1`)를 여기서는 **4,800 step 만 이어서** 학습합니다.

### (a) 보상 설정 확인 (envs1)

env1 은 정규화 패널티·걸음새 보상·생존 보상이 모두 0 이고 `allow_calf_contact` 도 켜져 있어
(정강이 접촉 허용) 제약이 가장 적습니다. 이후 라운드에서 여기에 보상을 더해갑니다.


**Exp 1 — env1 : 속도만 추종**

`envs1.yaml` 핵심 설정:

```
# envs1.yaml
reward:
  linear_vel_tracking: 1.5
  angular_vel_tracking: 1.0
cost:                        # 전부 0
termination:
  allow_calf_contact: true   # 정강이 접촉 허용
```

In [31]:
display(show_code(f"{repo_dir}/src/envs1.yaml"))

### (b) env1 모델 이어서 학습한 후 확인

`pretrained_env1` 체크포인트를 **`envs1.yaml` 보상으로 4,800 step 이어서 학습**한 뒤,
학습된 모델의 보행을 영상으로 확인합니다.


In [32]:
r1_model = finetune(ROUND1_MODEL, ROUND1_CFG, run_tag="env1")
vp = rollout_and_video(r1_model, ROUND1_CFG, tag="env1_after")
round_videos.append((vp, "env1 (4,800 step 이어서 학습한 후)"))
show_video(vp)

[Go2MujocoEnv] hip/abad joint indices used for hip_spread penalty: [0, 3, 6, 9]
저장 위치: c:\Users\enban\OneDrive\Desktop\RL_tutorial-main/models/2026-06-20_23-39-25-env1
[Go2MujocoEnv] hip/abad joint indices used for hip_spread penalty: [0, 3, 6, 9]
[env1] load pretrained: c:\Users\enban\OneDrive\Desktop\RL_tutorial-main/models/pretrained_env1/best_model.zip
Logging to c:\Users\enban\OneDrive\Desktop\RL_tutorial-main/logs\2026-06-20_23-39-25-env1_0


c:\Users\enban\anaconda3\envs\go2\lib\site-packages\stable_baselines3\common\callbacks.py:414: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.subproc_vec_env.SubprocVecEnv object at 0x00000214D77DA620> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x00000214D77DAE00>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


-------------------------------------
| cost/                  |          |
|    action_norm         | 0        |
|    action_rate         | 0        |
|    foot_clearance      | 0        |
|    gait_enforcement    | 0        |
|    hip_spread          | 0        |
|    joint_acc           | 0        |
|    joint_lim           | 0        |
|    joint_pos_deviation | 0        |
|    termination         | 0.167    |
|    torque              | 0        |
|    vertical_vel        | 0        |
|    xy_angular_vel      | 0        |
| reward/                |          |
|    ang_vel             | 0.562    |
|    base_height_reward  | 0        |
|    feet_air_time       | 0        |
|    healthy             | 0        |
|    lin_vel             | 1.15     |
|    total               | 1.71     |
|    total_costs         | 0        |
|    total_rewards       | 1.71     |
| rollout/               |          |
|    ep_len_mean         | 405      |
|    ep_rew_mean         | 648      |
| time/     

---

## 4. 라운드 2 — env1 → env2 (정규화·걸음새 보상 도입)

`env2` 보상으로 **거의 끝까지(~2,000,000 step) 미리 학습해 둔 체크포인트**(`pretrained_env2`)를
여기서는 **4,800 step 만 이어서** 학습합니다.

### (a) 보상 설정 확인 (envs1 / envs2)

이번 라운드에서 비교/사용할 두 보상 설정을 펼쳐 보고, 바뀐 항목을 요약합니다.


In [33]:
display(show_code(f"{repo_dir}/src/envs1.yaml"))
display(show_code(f"{repo_dir}/src/envs2.yaml"))

**env1 → env2 : 정규화·걸음새 보상 도입**

env1 은 속도 추종(`linear/angular_vel_tracking`)만 보는 *task-only* 설정입니다.
env2 는 여기에 다음을 더해, 마구잡이 동작을 억제하고 안정적인 trot 보행을 유도합니다.

- **정규화 패널티 추가** (모두 0 → 양수): `torque`, `vertical_vel`, `xy_angular_vel`,
  `action_rate`, `joint_limit`, `joint_acc`, `action_norm`, `joint_pos_deviation`
- **걸음새/발**: `gait_enforcement`(trot 패턴 강제), `foot_clearance`(스윙 시 발 높이) 도입
- **종료 조건**: `allow_calf_contact` `true → false` (정강이가 바닥에 닿으면 에피소드 종료)

아래 셀에서 실제로 바뀐 항목과 값을 출력합니다.


**정규화·걸음새 페널티 도입 (추가 항목 0 → 값)**

| 추가 항목 (0 → 값) | 가중치 | 효과 / 목적 |
|---|---|---|
| `torque` · `joint_acc` | 0.001 · 2.5e-7 | 에너지·충격 억제 |
| `action_rate` | 0.01 | 채터링·진동 억제 |
| `vertical_vel`, `xy_angular_vel` | 1.0, 0.2 | 몸통 출렁임·흔들림을 줄여 자세 안정 |
| `joint_limit` | 10.0 | 가동범위 초과 시 페널티 |
| `action_norm`, `joint_pos_deviation` | 0.005, 0.05 | 기준 자세에서 과도한 이탈 억제 |
| `gait_enforcement` | 0.05 | trot 위상 강제 → 대각 발 쌍의 접촉 타이밍 일치 |
| `foot_clearance` | 50 | 스윙 발 높이를 8 cm 목표로 유지 |
| `allow_calf_contact` | true → false | 정강이 접촉 = 종료 |

In [34]:
reward_diff(ROUND1_CFG, ROUND2_CFG)   # env1 -> env2

보상/설정 변화: envs1.yaml -> envs2.yaml
  cost.torque               : 0.0  ->  0.001
  cost.vertical_vel         : 0.0  ->  1.0
  cost.xy_angular_vel       : 0.0  ->  0.2
  cost.action_rate          : 0.0  ->  0.01
  cost.joint_limit          : 0.0  ->  10.0
  cost.joint_acc            : 0.0  ->  2.5e-07
  cost.action_norm          : 0.0  ->  0.005
  cost.joint_pos_deviation  : 0.0  ->  0.05
  cost.gait_enforcement     : 0.0  ->  0.05
  cost.foot_clearance       : 0.0  ->  50
  termination.allow_calf_contact: True  ->  False


[('cost.torque', 0.0, 0.001),
 ('cost.vertical_vel', 0.0, 1.0),
 ('cost.xy_angular_vel', 0.0, 0.2),
 ('cost.action_rate', 0.0, 0.01),
 ('cost.joint_limit', 0.0, 10.0),
 ('cost.joint_acc', 0.0, 2.5e-07),
 ('cost.action_norm', 0.0, 0.005),
 ('cost.joint_pos_deviation', 0.0, 0.05),
 ('cost.gait_enforcement', 0.0, 0.05),
 ('cost.foot_clearance', 0.0, 50),
 ('termination.allow_calf_contact', True, False)]

### (b) env2 모델 이어서 학습한 후 확인

`pretrained_env2` 체크포인트를 **`envs2.yaml` 보상으로 4,800 step 이어서 학습**한 뒤,
학습된 모델의 보행을 영상으로 확인합니다.


**Exp 2 — env1 ↔ env2 보행 비교**

- **관찰 포인트**: ① FR+RL 쌍 / FL+RR 쌍의 움직임  ② 스윙 시 발 높이  ③ 몸통 출렁임
- **트레이드오프**: 페널티 ↑ → 동작 절제 ↑, 속도 추종 적극성 ↓

In [35]:
r2_model = finetune(ROUND2_MODEL, ROUND2_CFG, run_tag="env2")
vp = rollout_and_video(r2_model, ROUND2_CFG, tag="env2_after")
round_videos.append((vp, "env2 (4,800 step 이어서 학습한 후)"))
show_video(vp)

[Go2MujocoEnv] hip/abad joint indices used for hip_spread penalty: [0, 3, 6, 9]
저장 위치: c:\Users\enban\OneDrive\Desktop\RL_tutorial-main/models/2026-06-20_23-39-47-env2
[Go2MujocoEnv] hip/abad joint indices used for hip_spread penalty: [0, 3, 6, 9]
[env2] load pretrained: c:\Users\enban\OneDrive\Desktop\RL_tutorial-main/models/pretrained_env2/best_model.zip
Logging to c:\Users\enban\OneDrive\Desktop\RL_tutorial-main/logs\2026-06-20_23-39-47-env2_0


c:\Users\enban\anaconda3\envs\go2\lib\site-packages\stable_baselines3\common\callbacks.py:414: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.subproc_vec_env.SubprocVecEnv object at 0x00000214D74E3D00> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x00000214D74E2C50>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


-------------------------------------
| cost/                  |          |
|    action_norm         | 0.0104   |
|    action_rate         | 0.00722  |
|    foot_clearance      | 0.132    |
|    gait_enforcement    | 0.155    |
|    hip_spread          | 0        |
|    joint_acc           | 0.0206   |
|    joint_lim           | 0.000702 |
|    joint_pos_deviation | 0.0992   |
|    termination         | 0.0208   |
|    torque              | 0.341    |
|    vertical_vel        | 0.00585  |
|    xy_angular_vel      | 0.0827   |
| reward/                |          |
|    ang_vel             | 0.938    |
|    base_height_reward  | 0        |
|    feet_air_time       | 0        |
|    healthy             | 0        |
|    lin_vel             | 1.14     |
|    total               | 1.22     |
|    total_costs         | 0.854    |
|    total_rewards       | 2.08     |
| rollout/               |          |
|    ep_len_mean         | 541      |
|    ep_rew_mean         | 626      |
| time/     

---

## 5. 라운드 3 — env2 → env3 (positive shaping 추가)

`env3` 보상으로 ~2,000,000 step 미리 학습해 둔 체크포인트(`pretrained_env3`)를
여기서는 4,800 step 만 이어서 학습합니다.

### (a) 보상 설정 확인 (envs2 / envs3)

env2 대비 보상 설정을 펼쳐 보고, 바뀐 항목을 요약합니다.


In [36]:
display(show_code(f"{repo_dir}/src/envs2.yaml"))
display(show_code(f"{repo_dir}/src/envs3.yaml"))

**env2 → env3 : 자세·생존 보상(positive shaping) 추가**

env2 의 패널티 위주 설정에, 좋은 자세를 직접 장려하는 양(+)의 보상을 더하고
일부 패널티 강도를 미세 조정합니다.

- **positive shaping 추가** (0 → 양수): `healthy`(생존), `base_height`(기준 높이 유지),
  `feet_air_time`(발 체공 시간)
- **패널티 완화**: `vertical_vel`(1.0→0.8), `xy_angular_vel`(0.2→0.15),
  `gait_enforcement`(0.05→0.04), `foot_clearance`(50→25)
- **패널티 강화**: `action_norm`(0.005→0.007), `joint_pos_deviation`(0.05→0.08)

아래 셀에서 실제로 바뀐 항목과 값을 출력합니다.


**Positive Shaping 추가 및 미세 조정**

① Positive shaping 추가 (0 → 값)

| 항목 | 값: 설명 |
|---|---|
| `healthy` | 1.0: 생존 자체에 보상 |
| `base_height` | 0.01: 몸통 높이 0.36 m 유지 |
| `feet_air_time` | 0.3: 체공 시간이 0.5 s 를 넘는 보폭을 장려 |

② 페널티 미세 조정

| 구분 | 변화 |
|---|---|
| 완화 ↓ | `vertical_vel` 1.0→0.8, `xy_ang` 0.2→0.15, `gait` 0.05→0.04, `foot_clear` 50→25 |
| 강화 ↑ | `action_norm` 0.005→0.007, `joint_pos_dev` 0.05→0.08 |

- 벌점 위주 설계 → 양의 보상 추가 → 좋은 자세
- 실전 튜닝: 영상 확인 → 가중치 1~2개 미세 조정 → 재학습 반복

In [37]:
reward_diff(ROUND2_CFG, ROUND3_CFG)   # env2 -> env3

보상/설정 변화: envs2.yaml -> envs3.yaml
  reward.healthy            : 0.0  ->  1.0
  reward.base_height        : 0.0  ->  0.01
  reward.feet_air_time      : 0.0  ->  0.3
  cost.vertical_vel         : 1.0  ->  0.8
  cost.xy_angular_vel       : 0.2  ->  0.15
  cost.action_norm          : 0.005  ->  0.007
  cost.joint_pos_deviation  : 0.05  ->  0.08
  cost.gait_enforcement     : 0.05  ->  0.04
  cost.foot_clearance       : 50  ->  25


[('reward.healthy', 0.0, 1.0),
 ('reward.base_height', 0.0, 0.01),
 ('reward.feet_air_time', 0.0, 0.3),
 ('cost.vertical_vel', 1.0, 0.8),
 ('cost.xy_angular_vel', 0.2, 0.15),
 ('cost.action_norm', 0.005, 0.007),
 ('cost.joint_pos_deviation', 0.05, 0.08),
 ('cost.gait_enforcement', 0.05, 0.04),
 ('cost.foot_clearance', 50, 25)]

### (b) env3 모델 이어서 학습한 후 확인

`pretrained_env3` 체크포인트를 **`envs3.yaml` 보상으로 4,800 step 이어서 학습**한 뒤,
학습된 모델의 보행을 영상으로 확인합니다.


**Exp 3 — env2 ↔ env3 보행 비교**

env2(패널티 위주)와 env3(positive shaping 추가)의 보행을 나란히 비교해 자세·안정성 변화를 확인합니다.

In [38]:
r3_model = finetune(ROUND3_MODEL, ROUND3_CFG, run_tag="env3")
vp = rollout_and_video(r3_model, ROUND3_CFG, tag="env3_after")
round_videos.append((vp, "env3 (4,800 step 이어서 학습한 후)"))
show_video(vp)

[Go2MujocoEnv] hip/abad joint indices used for hip_spread penalty: [0, 3, 6, 9]
저장 위치: c:\Users\enban\OneDrive\Desktop\RL_tutorial-main/models/2026-06-20_23-40-10-env3
[Go2MujocoEnv] hip/abad joint indices used for hip_spread penalty: [0, 3, 6, 9]
[env3] load pretrained: c:\Users\enban\OneDrive\Desktop\RL_tutorial-main/models/pretrained_env3/best_model.zip
Logging to c:\Users\enban\OneDrive\Desktop\RL_tutorial-main/logs\2026-06-20_23-40-10-env3_0


c:\Users\enban\anaconda3\envs\go2\lib\site-packages\stable_baselines3\common\callbacks.py:414: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.subproc_vec_env.SubprocVecEnv object at 0x00000214D2A18C40> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x00000214D2A19630>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


-------------------------------------
| cost/                  |          |
|    action_norm         | 0.0146   |
|    action_rate         | 0.00717  |
|    foot_clearance      | 0.0229   |
|    gait_enforcement    | 0.06     |
|    hip_spread          | 0        |
|    joint_acc           | 0.0251   |
|    joint_lim           | 0.00168  |
|    joint_pos_deviation | 0.0635   |
|    termination         | 0.0208   |
|    torque              | 0.277    |
|    vertical_vel        | 0.00633  |
|    xy_angular_vel      | 0.058    |
| reward/                |          |
|    ang_vel             | 0.96     |
|    base_height_reward  | 0.0058   |
|    feet_air_time       | -0.0171  |
|    healthy             | 1        |
|    lin_vel             | 1.43     |
|    total               | 2.85     |
|    total_costs         | 0.536    |
|    total_rewards       | 3.38     |
| rollout/               |          |
|    ep_len_mean         | 626      |
|    ep_rew_mean         | 1.8e+03  |
| time/     

---

## 6. 결과 비교

env1 / env2 / env3 를 각각 4,800 step 이어서 학습한 결과 보행을 나란히 비교합니다.


In [39]:
import base64

def _embed(path, caption, width=320):
    b64 = base64.b64encode(open(path, "rb").read()).decode("ascii")
    return f"""
    <figure style="margin:0; text-align:center;">
      <video controls autoplay muted loop width="{width}">
        <source src="data:video/mp4;base64,{b64}" type="video/mp4">
      </video>
      <figcaption style="margin-top:6px; font-size:12px;">{caption}</figcaption>
    </figure>
    """

display(HTML(
    '<div style="display:flex; gap:12px; flex-wrap:wrap;">'
    + "".join(_embed(p, c) for p, c in round_videos if os.path.exists(p))
    + "</div>"
))

---

## 7. TensorBoard 로 학습 로그 보기

각 라운드(env1/env2/env3) 학습의 보상/손실 곡선을 TensorBoard 로 비교합니다.
런(run)별로 `logs/` 아래에 저장되며, 셀을 실행하면 노트북 안에 대시보드가 뜹니다.

> `logs/pretrained_env*` 에 사전학습(0→~2M step) 로그가 포함돼 있어, **사전학습 전체 곡선 + 이어학습 구간**이 같은 step 축 위에 함께 표시됩니다.


In [40]:
# 학습 로그 시각화 (인라인 TensorBoard)
import os
os.chdir(repo_dir)          # logs 상대경로 기준 보장
%load_ext tensorboard
%tensorboard --logdir logs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 35112), started 0:06:14 ago. (Use '!kill 35112' to kill it.)